<a href="https://colab.research.google.com/github/andrew23n/TennisPredictions/blob/main/TennisPredictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

np.random.seed(42)
pd.set_option('display.max_columns', None)

In [ ]:
#full github of data
#https://github.com/Aneeshers/tennis-sackmann-archive

years = range(2000,2024)
dfs = []

for year in years:
  url = f"https://raw.githubusercontent.com/Aneeshers/tennis-sackmann-archive/refs/heads/main/atp/atp_matches_{year}.csv"

  dfs.append(pd.read_csv(url))

df = pd.concat(dfs, ignore_index=True)

In [ ]:
df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_seed,loser_entry,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2000-301,Auckland,Hard,32,A,20000110,1,103163,1.0,NaN,Tommy Haas,R,188.0,GER,21.7,101543,NaN,NaN,Jeff Tarango,L,180.0,USA,31.1,7-5 4-6 7-5,3,R32,108.0,18.0,4.0,96.0,49.0,39.0,28.0,17.0,3.0,5.0,7.0,8.0,106.0,55.0,39.0,29.0,17.0,4.0,7.0,11.0,1612.0,63.0,595.0
1,2000-301,Auckland,Hard,32,A,20000110,2,102607,NaN,Q,Juan Balcells,R,190.0,ESP,24.5,102644,NaN,NaN,Franco Squillari,L,183.0,ARG,24.3,7-5 7-5,3,R32,85.0,5.0,3.0,76.0,52.0,39.0,13.0,12.0,5.0,6.0,5.0,10.0,74.0,32.0,25.0,18.0,12.0,3.0,6.0,211.0,157.0,49.0,723.0
2,2000-301,Auckland,Hard,32,A,20000110,3,103252,NaN,NaN,Alberto Martin,R,175.0,ESP,21.3,102238,NaN,NaN,Alberto Berasategui,R,173.0,ESP,26.5,6-3 6-1,3,R32,56.0,0.0,0.0,55.0,35.0,25.0,12.0,8.0,1.0,1.0,0.0,6.0,56.0,33.0,20.0,7.0,8.0,7.0,11.0,48.0,726.0,59.0,649.0
3,2000-301,Auckland,Hard,32,A,20000110,4,103507,7.0,NaN,Juan Carlos Ferrero,R,183.0,ESP,19.9,103819,NaN,NaN,Roger Federer,R,185.0,SUI,18.4,6-4 6-4,3,R32,68.0,5.0,1.0,53.0,28.0,26.0,15.0,10.0,0.0,0.0,11.0,2.0,70.0,43.0,29.0,14.0,10.0,6.0,8.0,45.0,768.0,61.0,616.0
4,2000-301,Auckland,Hard,32,A,20000110,5,102103,NaN,Q,Michael Sell,R,180.0,USA,27.3,102765,4.0,NaN,Nicolas Escude,R,185.0,FRA,23.7,0-6 7-6(7) 6-1,3,R32,115.0,1.0,2.0,98.0,66.0,39.0,14.0,13.0,6.0,11.0,8.0,8.0,92.0,46.0,34.0,18.0,12.0,5.0,9.0,167.0,219.0,34.0,873.0


In [46]:
filtered_df = df.drop(columns=["score", "minutes", "w_ace", "w_df", "w_svpt", "w_1stIn", "w_1stWon", "w_2ndWon", "w_SvGms", "w_bpSaved", "w_bpFaced", "l_ace", "l_df", "l_svpt", "l_1stIn", "l_1stWon", "l_2ndWon", "l_SvGms", "l_bpSaved", "l_bpFaced"])
filtered_df["tourney_date"] = pd.to_datetime(filtered_df["tourney_date"], format="%Y%m%d")
filtered_df = filtered_df.sort_values(["tourney_date", "tourney_id", "match_num"]).reset_index(drop=True)

In [47]:
filtered_df.head()
print(filtered_df["tourney_date"].is_monotonic_increasing)

True


In [48]:
# Custom field to track player winrates
player_wins = {}
player_matches = {}

winner_win_rate = []
loser_win_rate = []

for _, row in filtered_df.iterrows():
  winner = row["winner_id"]
  loser = row["loser_id"]

  winner_wins = player_wins.get(winner, 0)
  winner_matches = player_matches.get(winner, 0)

  loser_wins = player_wins.get(loser,0)
  loser_matches = player_matches.get(loser,0)

  #0.5 is neutral winrate
  winner_rate = (winner_wins / winner_matches if winner_matches > 0 else 0.5)
  loser_rate = (loser_wins / loser_matches if loser_matches > 0 else 0.5)

  winner_win_rate.append(winner_rate)
  loser_win_rate.append(loser_rate)

  player_wins[winner] = winner_wins + 1
  player_matches[winner] = winner_matches + 1

  player_wins[loser] = loser_wins
  player_matches[loser] = loser_matches + 1

filtered_df["winner_win_rate"] = winner_win_rate
filtered_df["loser_win_rate"] = loser_win_rate
filtered_df["win_rate_diff"] = filtered_df["winner_win_rate"] - filtered_df["loser_win_rate"]


In [49]:
# Custom field for recent form: last 10 matches
from collections import deque

player_recent_results = {}

winner_recent_win_rate = []
loser_recent_win_rate = []

for _, row in filtered_df.iterrows():
  winner = row["winner_id"]
  loser = row["loser_id"]

  #maxlen ensures only 10 matches at a time
  winner_history = player_recent_results.get(winner, deque(maxlen=10))
  loser_history = player_recent_results.get(loser, deque(maxlen=10))

  if len(winner_history) > 0:
    winner_rate = sum(winner_history) / len(winner_history)
  else:
    winner_rate = 0.5

  if len(loser_history) > 0:
    loser_rate = sum(loser_history) / len(loser_history)
  else:
    loser_rate = 0.5

  winner_recent_win_rate.append(winner_rate)
  loser_recent_win_rate.append(loser_rate)

  winner_history.append(1)
  loser_history.append(0)

  player_recent_results[winner] = winner_history
  player_recent_results[loser] = loser_history

filtered_df["winner_recent_win_rate"] = winner_recent_win_rate
filtered_df["loser_recent_win_rate"] = loser_recent_win_rate
filtered_df["recent_win_rate_diff"] = filtered_df["winner_recent_win_rate"] - filtered_df["loser_recent_win_rate"]

In [50]:
# Custom field for recent form: last 5 matches

player_recent_5 = {}

winner_recent_5_win_rate = []
loser_recent_5_win_rate = []

for _, row in filtered_df.iterrows():

    winner = row["winner_id"]
    loser = row["loser_id"]

    winner_history = player_recent_5.get(winner, deque(maxlen=5))

    loser_history = player_recent_5.get(loser, deque(maxlen=5))

    winner_rate = (sum(winner_history) / len(winner_history) if len(winner_history) > 0 else 0.5)
    loser_rate = (sum(loser_history) / len(loser_history) if len(loser_history) > 0 else 0.5)

    winner_recent_5_win_rate.append(winner_rate)
    loser_recent_5_win_rate.append(loser_rate)

    winner_history.append(1)
    loser_history.append(0)

    player_recent_5[winner] = winner_history
    player_recent_5[loser] = loser_history

filtered_df["winner_recent_5_win_rate"] = winner_recent_5_win_rate
filtered_df["loser_recent_5_win_rate"] = loser_recent_5_win_rate

filtered_df["recent_5_win_rate_diff"] = filtered_df["winner_recent_5_win_rate"] - filtered_df["loser_recent_5_win_rate"]

In [51]:
# Original matches: winner is Player 1
df1 = filtered_df.copy()

df1["rank_diff"] = df1["winner_rank"] - df1["loser_rank"]
df1["rank_points_diff"] = df1["winner_rank_points"] - df1["loser_rank_points"]
df1["age_diff"] = df1["winner_age"] - df1["loser_age"]
df1["height_diff"] = df1["winner_ht"] - df1["loser_ht"]

df1["target"] = 1

df1["win_rate_diff"] = df1["winner_win_rate"] - df1["loser_win_rate"]
df1["recent_win_rate_diff"] = df1["winner_recent_win_rate"] - df1["loser_recent_win_rate"]
df1["recent_5_win_rate_diff"] = df1["winner_recent_5_win_rate"] - df1["loser_recent_5_win_rate"]


# Reversed matches: loser is Player 1
df2 = filtered_df.copy()

df2["rank_diff"] = df2["loser_rank"] - df2["winner_rank"]
df2["rank_points_diff"] = df2["loser_rank_points"] - df2["winner_rank_points"]
df2["age_diff"] = df2["loser_age"] - df2["winner_age"]
df2["height_diff"] = df2["loser_ht"] - df2["winner_ht"]

df2["target"] = 0

df2["win_rate_diff"] = df2["winner_win_rate"] - df2["loser_win_rate"]
df2["recent_win_rate_diff"] = df2["loser_recent_win_rate"] - df2["winner_recent_win_rate"]
df2["recent_5_win_rate_diff"] = df2["winner_recent_5_win_rate"] - df2["loser_recent_5_win_rate"]

model_df = pd.concat([df1, df2], ignore_index=True)

In [ ]:
filtered_df.head(10)

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_seed,loser_entry,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,best_of,round,winner_rank,winner_rank_points,loser_rank,loser_rank_points,winner_win_rate,loser_win_rate,win_rate_diff,winner_recent_win_rate,loser_recent_win_rate,recent_win_rate_diff
0,2000-301,Auckland,Hard,32,A,2000-01-10,1,103163,1.0,NaN,Tommy Haas,R,188.0,GER,21.7,101543,NaN,NaN,Jeff Tarango,L,180.0,USA,31.1,3,R32,11.0,1612.0,63.0,595.0,0.5,0.5,0.0,0.5,0.5,0.0
1,2000-301,Auckland,Hard,32,A,2000-01-10,2,102607,NaN,Q,Juan Balcells,R,190.0,ESP,24.5,102644,NaN,NaN,Franco Squillari,L,183.0,ARG,24.3,3,R32,211.0,157.0,49.0,723.0,0.5,0.5,0.0,0.5,0.5,0.0
2,2000-301,Auckland,Hard,32,A,2000-01-10,3,103252,NaN,NaN,Alberto Martin,R,175.0,ESP,21.3,102238,NaN,NaN,Alberto Berasategui,R,173.0,ESP,26.5,3,R32,48.0,726.0,59.0,649.0,0.5,0.5,0.0,0.5,0.5,0.0
3,2000-301,Auckland,Hard,32,A,2000-01-10,4,103507,7.0,NaN,Juan Carlos Ferrero,R,183.0,ESP,19.9,103819,NaN,NaN,Roger Federer,R,185.0,SUI,18.4,3,R32,45.0,768.0,61.0,616.0,0.5,0.5,0.0,0.5,0.5,0.0
4,2000-301,Auckland,Hard,32,A,2000-01-10,5,102103,NaN,Q,Michael Sell,R,180.0,USA,27.3,102765,4.0,NaN,Nicolas Escude,R,185.0,FRA,23.7,3,R32,167.0,219.0,34.0,873.0,0.5,0.5,0.0,0.5,0.5,0.0
5,2000-301,Auckland,Hard,32,A,2000-01-10,6,102021,NaN,NaN,Michael Chang,R,175.0,USA,27.8,101647,NaN,NaN,Byron Black,R,175.0,ZIM,30.2,3,R32,50.0,722.0,70.0,563.0,0.5,0.5,0.0,0.5,0.5,0.0
6,2000-301,Auckland,Hard,32,A,2000-01-10,7,101320,NaN,NaN,Magnus Gustafsson,R,185.0,SWE,33.0,103066,NaN,WC,Mark Nielsen,R,185.0,NZL,22.2,3,R32,60.0,626.0,246.0,135.0,0.5,0.5,0.0,0.5,0.5,0.0
7,2000-301,Auckland,Hard,32,A,2000-01-10,8,102563,5.0,NaN,Thomas Johansson,R,180.0,SWE,24.7,102785,NaN,Q,Glenn Weiner,R,188.0,USA,23.7,3,R32,43.0,785.0,334.0,88.0,0.5,0.5,0.0,0.5,0.5,0.0
8,2000-301,Auckland,Hard,32,A,2000-01-10,9,102854,6.0,NaN,Sjeng Schalken,R,193.0,NED,23.3,101964,NaN,NaN,Goran Ivanisevic,L,193.0,CRO,28.3,3,R32,37.0,843.0,68.0,571.0,0.5,0.5,0.0,0.5,0.5,0.0
9,2000-301,Auckland,Hard,32,A,2000-01-10,10,102494,NaN,Q,Tomas Behrend,R,193.0,GER,25.0,103082,NaN,NaN,Markus Hantschk,R,188.0,GER,22.1,3,R32,115.0,344.0,95.0,416.0,0.5,0.5,0.0,0.5,0.5,0.0


In [52]:
features = [
    "rank_diff",
    "rank_points_diff",
    "age_diff",
    "height_diff",
    "win_rate_diff",
    "recent_win_rate_diff",
    "recent_5_win_rate_diff"
]

model_df = model_df.dropna(subset=features).copy()

In [53]:
#training dataframe -> 2000-2018
training_df = model_df[model_df["tourney_date"].dt.year <= 2018]

#validation dataframe -> 2019-2021
validation_df = model_df[(model_df["tourney_date"].dt.year >= 2019) & (model_df["tourney_date"].dt.year <= 2021)]

#test dataframe -> 2022+
test_df = model_df[model_df["tourney_date"].dt.year >= 2022]

In [54]:
X_train = training_df[features]
y_train = training_df["target"]

X_val = validation_df[features]
y_val = validation_df["target"]

X_test = test_df[features]
y_test = test_df["target"]

In [55]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [56]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [57]:
y_val_pred = model.predict(X_val_scaled)
y_val_prob = model.predict_proba(X_val_scaled)[:,1]

print(y_val_pred)
print(y_val_prob)

rank_baseline_pred = (X_val["rank_diff"] < 0).astype(int)
rank_accuracy = accuracy_score(y_val, rank_baseline_pred)

[0 1 1 ... 0 0 0]
[0.48857399 0.55577163 0.53024537 ... 0.05512425 0.09229744 0.07050461]


In [58]:
accuracy = accuracy_score(y_val, y_val_pred)
loss = log_loss(y_val, y_val_prob)
auc = roc_auc_score(y_val, y_val_prob)

print(f'Baseline Accuracy: {rank_accuracy:.8f}')
print(f'Accuracy: {accuracy:.8f}')
print(f'Loss: {loss:.8f}')
print(f'AUC: {auc:.8f}')

Baseline Accuracy: 0.62745098
Accuracy: 0.64381542
Loss: 0.62852061
AUC: 0.70195380


In [59]:
from sklearn.ensemble import RandomForestClassifier

rfc_model = RandomForestClassifier(n_estimators = 100, max_depth = 8, n_jobs=-1, random_state=42)

rfc_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=8, n_jobs=-1, random_state=42)

In [60]:
rfc_pred = rfc_model.predict(X_val)
rfc_prob = rfc_model.predict_proba(X_val)[:,-1]

rfc_accuracy = accuracy_score(y_val, rfc_pred)
rfc_loss = log_loss(y_val, rfc_prob)
rfc_auc = roc_auc_score(y_val, rfc_prob)

print(f"Random Forest Accuracy: {rfc_accuracy:.8f}")
print(f"Random Forest Log Loss: {rfc_loss:.8f}")
print(f"Random Forest AUC: {rfc_auc:.8f}")

Random Forest Accuracy: 0.84328468
Random Forest Log Loss: 0.37054311
Random Forest AUC: 0.92421006


In [61]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=300, max_depth = 4, learning_rate=0.05, subsample=0.8,colsample_bytree=0.8, n_jobs = -1, random_state=42)

xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=-1, num_parallel_tree=None, ...)

In [62]:
xgb_pred = xgb_model.predict(X_val)
xgb_prob = xgb_model.predict_proba(X_val)[:, 1]

xgb_accuracy = accuracy_score(y_val, xgb_pred)

xgb_loss = log_loss(y_val, xgb_prob)

xgb_auc = roc_auc_score(y_val, xgb_prob)

print(f"XGBoost Accuracy: {xgb_accuracy:.8f}")
print(f"XGBoost Log Loss: {xgb_loss:.8f}")
print(f"XGBoost AUC: {xgb_auc:.8f}")

XGBoost Accuracy: 0.84645437
XGBoost Log Loss: 0.32379245
XGBoost AUC: 0.93412857


In [63]:
player_id = 104745  # example

print(
    filtered_df[
        (filtered_df["winner_id"] == player_id) |
        (filtered_df["loser_id"] == player_id)
    ][
        [
            "tourney_date",
            "winner_name",
            "loser_name",
            "winner_recent_5_win_rate",
            "loser_recent_5_win_rate"
        ]
    ].head(15)
)

      tourney_date          winner_name          loser_name  \
7928    2002-04-29         Rafael Nadal       Ramon Delgado   
7941    2002-04-29       Olivier Rochus        Rafael Nadal   
11004   2003-04-14         Rafael Nadal        Karol Kucera   
11025   2003-04-14         Rafael Nadal        Albert Costa   
11035   2003-04-14      Guillermo Coria        Rafael Nadal   
11057   2003-04-21         Rafael Nadal  Juan Antonio Marin   
11077   2003-04-21        Alex Corretja        Rafael Nadal   
11286   2003-05-12         Rafael Nadal  Paul Henri Mathieu   
11303   2003-05-12         Rafael Nadal         Carlos Moya   
11311   2003-05-12        Gaston Gaudio        Rafael Nadal   
11664   2003-06-23         Rafael Nadal         Mario Ancic   
11723   2003-06-23         Rafael Nadal          Lee Childs   
11752   2003-06-23  Paradorn Srichaphan        Rafael Nadal   
11854   2003-07-07         Rafael Nadal   Younes El Aynaoui   
11864   2003-07-07         Rafael Nadal       Albert Po